# Week 8: Web Scraping Part 2
**Name**: Isabelle Sakelaris \
**Date**: October 19, 2026

In [1]:
import requests

url = "https://www.gutenberg.org/cache/epub/22403/pg22403-images.html"
headers = {"User-Agent": "INFO664-class-project (educational)"}

response = requests.get(url, headers=headers)

with open("gutenberg.html", "w", encoding="utf-8") as file:
    file.write(response.text)

In [2]:
import csv
import json # allows us to write json files and interpret requests as JSON
import os # operating system library, let's us do things like making new folders
from time import sleep
from bs4 import BeautifulSoup # the scraping library!

In [3]:
SCRAPED_PAGES_DIRECTORY = "scraped_pages/"
SCRAPED_PAGES_PREFIX = "page"
scraped_pages_count = 0 # keep track of how many pages we have scraped

headers = {"User-Agent": "INFO664-class-project (educational)"}   # you can call this whatever you want, it shows up on the scraped website's logs as who you are (ie not a Google/AI bot or web browser)

os.makedirs("scraped_pages", exist_ok=True)  # make a new folder to hold our scraped website HTML data. exist_ok = True means, if the directory already exists, do nothing and don't return an error

def fetch_page(url):  # declaring a new function, which takes a URL to be scraped, and a destination filename, as arguments
    global scraped_pages_count  # use the scraped_pages_count from the top of the script, instead of making a new local one
    filename = f"{SCRAPED_PAGES_DIRECTORY}{SCRAPED_PAGES_PREFIX}{scraped_pages_count}.html"
    if os.path.exists(filename):  # check to see if file already exists, if so, don't download it from the web, load it instead
        scraped_pages_count = scraped_pages_count + 1
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()  # in this case, return provides the read file data, and escapes from the function, ignoring everything below

    response = requests.get(url, headers=headers) # grab the file, using our pre-defined headers as the header data

    if not response.ok: # if the response reports an error
        print(url, "failed with status code", response.status_code) # tell us why it failed
        return None # return a "None" object, since there's nothing usable in the response, and escape from this function
    with open(filename, "w",  encoding="utf-8") as file:
        file.write(response.text)
        scraped_pages_count = scraped_pages_count + 1
    sleep(10) # wait 10 seconds, since our robots.txt file asks for a 10 second Crawl-delay
    return response.text

In [4]:
url = "https://www.gutenberg.org/cache/epub/22403/pg22403-images.html"
html = fetch_page(url) # filling the var "html" with the actual html, as text
soup = BeautifulSoup(html, "html.parser") # new var, converting the text into something structured using the Beautiful Soup library

In [5]:
poem_titles = []
title_element = soup.find_all(name="i")
br = "<br>"
for title in title_element:
    if title.get_text(strip=True)[0].isdigit():
        poem_title = "".join(filter(lambda c: c.isalpha() or c.isspace(), title.get_text())) # Filter out non-letter characters, found this on stack overflow
        poem_titles.append(poem_title)
print(poem_titles)

['', '', '\n\nFor a Picture of\n\nSt Dorothea', '\n\nHeavenHaven\n\nA nun takes the veil', '\nThe Habit of Perfection', '\n\nTHE WRECK\n\nOF THE DEUTSCHLAND', '\nPenmaen Pool', '\n\nThe Silver Jubilee\n\nTo James First Bishop of Shrewsbury on the th Year\n\nof his Episcopate July  ', '\nGods Grandeur', '\nThe Starlight Night', '\nSpring', '\nThe Lantern out of Doors', '\nThe Sea and the Skylark', '\nPied Beauty', '\nHurrahing in Harvest', '\nCaged Skylark', '\nIn the Valley of the Elwy', '\nThe May Magnificat', '\nDuns Scotuss Oxford', '\nHenry Purcell', '\nPeace', '\nMorning Midday and Evening Sacrifice', '\nAndromeda', '\nThe Candle Indoors', '\nFelix Randal', '\nBrothers', '\nSpelt from Sibyls Leaves', '\nInversnaid', '\nRibblesdale', '\n\nThe Blessed Virgin compared to the Air we\n\nBreathe', '\nTo what serves Mortal Beauty', '\nThe Soldier', '\nCarrion Comfort', '', '\nHarry Ploughman', '', '', '', '', '\nThat Nature is a Heraclitean Fire and of\nthe comfort of the Resurrection', 

In [6]:
new_titles = []
for title in poem_titles:
    text_title = title.replace("\n", "")
    new_titles.append(text_title)
new_titles = list(filter(None, new_titles)) #found this on stack overflow
print(new_titles)

['For a Picture ofSt Dorothea', 'HeavenHavenA nun takes the veil', 'The Habit of Perfection', 'THE WRECKOF THE DEUTSCHLAND', 'Penmaen Pool', 'The Silver JubileeTo James First Bishop of Shrewsbury on the th Yearof his Episcopate July  ', 'Gods Grandeur', 'The Starlight Night', 'Spring', 'The Lantern out of Doors', 'The Sea and the Skylark', 'Pied Beauty', 'Hurrahing in Harvest', 'Caged Skylark', 'In the Valley of the Elwy', 'The May Magnificat', 'Duns Scotuss Oxford', 'Henry Purcell', 'Peace', 'Morning Midday and Evening Sacrifice', 'Andromeda', 'The Candle Indoors', 'Felix Randal', 'Brothers', 'Spelt from Sibyls Leaves', 'Inversnaid', 'Ribblesdale', 'The Blessed Virgin compared to the Air weBreathe', 'To what serves Mortal Beauty', 'The Soldier', 'Carrion Comfort', 'Harry Ploughman', 'That Nature is a Heraclitean Fire and ofthe comfort of the Resurrection', 'In honour ofSt Alphonsus RodriguezLaybrother of the Society of Jesus', 'To R B', 'Summa', 'Ashboughs', 'Cheery Beggar', 'Moonrise

Found the titles! See above.

[<p id="id00059">Shape nothing, lips; be lovely-dumb:<br/>
 
 It is the shut, the curfew sent<br/>
 
 From there where all surrenders come<br/>
 
 Which only makes you eloquent.<br/>
 </p>,
 <p id="id00060">Be shellèd, eyes, with double dark<br/>
 
 And find the uncreated light:<br/>
 
 This ruck and reel which you remark<br/>
 
 Coils, keeps, and teases simple sight.<br/>
 </p>,
 <p id="id00061">Palate, the hutch of tasty lust,<br/>
 
 Desire not to be rinsed with wine:<br/>
 
 The can must be so sweet, the crust<br/>
 
 So fresh that come in fasts divine!<br/>
 </p>,
 <p id="id00062">Nostrils, your careless breath that spend<br/>
 
 Upon the stir and keep of pride,<br/>
 
 What relish shall the censers send<br/>
 
 Along the sanctuary side!<br/>
 </p>,
 <p id="id00063">O feel-of-primrose hands, O feet<br/>
 
 That want the yield of plushy sward,<br/>
 
 But you shall walk the golden street<br/>
 
 And you unhouse and house the Lord.<br/>
 </p>]

Now trying to figure out the stanzas.

In [7]:
p_elements = soup.find_all("p")
has_line_breaks = []

for p in p_elements:
    if "\n" in p.get_text():
        has_line_breaks.append(p.get_text())
print(len(has_line_breaks))

479


In [8]:
not_stanzas = soup.find_all(style="margin-top: 2em")

In [9]:
non_title_p = []
for p in p_elements:
    if non_title_p != p:
        non_title_p.append(p.get_text())

### Going back to a class example of how to create a dataset from scraped data

In [10]:
entry_titles = [] 
entry_paragraphs = [] #put something in here

for child in title_element:
    entry_titles.append(child.get_text())

entry_dictionary = {
    "title": entry_titles,
    "paragraphs": entry_paragraphs
}

In [11]:
# change the logic below for your own data...

#def scrape_data(html, url):
    # soup = BeautifulSoup(html, "html.parser") #convert the html input into a soup
    # title_element = soup.find(class_="entry-title")
    # entry_title - title_element.get_text() # creating a var containing the text of the title
    # entry_content_element - soup.find(class_="entry-content") # creating a var for every paragraph in the entry content
    # entry_paragraphs = []
    # for child in children:
        # entry_paragraphs.append(child.get_text()) # append the para to the list of "entry_paragraphs"

    # entry_dictionary = {
        # "title": entry_title,
        # "paragraphs": entry_paragraphs,
        # "url": url
       # }
    # return entry_dictionary
# print(scrape_data(html, url)) # this will confirm that the function is outputting a dictionary w the scraped data

In [12]:
#urls = [
   # "[a blog entry]",
   # "[a different blog entry]",
    #"[a third blog entry]"
# ]

# scraped_data = [] # will be an empty list for the dictionaries representing the scraped websites

# for u in urls: # for each of the urls in the list of urls...
    # html = fetch_page(u) # fetch the page, getting it from the web and loading it as an html string
    # website_data = scrape_data(html, u) # scrape the data in that string
    # scrape_data.append(website_data) #append that scraped data to the scraped_data list

# print(scraped_data)
# print(len(scraped_data)) # will tell you how many entries are in the dict!

To save a list of dictionaries as a json file we can load in other contexts, we use json.dump function!

In [13]:
# with open("scraped_data.json", "w", encoding="utf-8") as file: #create a new file and refer to it as "file"
   # json.dump(scraped_data, file, indent=4, ensure_ascii=False) # ensure_ascii = false makes sure that you don't strip the data of "non-conforming" characters 

### Homework
- In a notebook, list your data sources as properly-formatted markdown URLs. The links should go directly to the datasets, API documentation, or raw data that you intend to use

My data source is [this Project Gutenberg book](https://www.gutenberg.org/cache/epub/22403/pg22403-images.html).
  
- Outline the steps that you need to take to “wrangle” your dataset so that it is optimized for your particular intervention. This might involve processes of merging, sorting, filtering, cleaning, storing, or formatting your data so that you can accomplish your desired analysis using Python. Highlight any challenges you anticipate encountering as you wrangle your data. “I need to figure out how to …” or “I know how to x, but I don’t yet know how to y”.

I have already figured out how to wrangle the poem titles from the dataset (see above the "going back to class example section"), but I am having trouble getting the text of the poems. 

Every poem title and stanza is in a `<p>` with a squential numeric id, so there is very little to differentiate them. I need to figure out how to attribute different stanzas to a single poem, and to find where new poems start and end. I might be able to do that by looking at each sequential `<p>` and searching for the presence of a title (or the title styling).

I will also need to separate the stanzas into lines, but that probably won't be difficult once I figure out connecting the stanzas to the right poems. I can do that by splitting the `<p>` stanzas on line breaks. 
